# FinWatch — Exploratory Data Analysis

**Author:** Barbara Werobaobayi  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Purpose:** Understand the data before building the pipeline.

---

### Setup

Download the data first:
```
kaggle competitions download -c home-credit-default-risk
unzip home-credit-default-risk.zip -d data/raw/
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

DATA = Path('../data/raw')
print('Data files found:', [p.name for p in DATA.glob('*.csv')])

## 1. Load Main Application Table

In [ ]:
app = pd.read_csv(DATA / 'application_train.csv')
print(f'Shape: {app.shape}')
app.head(3)

## 2. Target Distribution

The original TARGET=1 means default. We repurpose this as a vulnerability proxy.

In [ ]:
target_counts = app['TARGET'].value_counts()
print(target_counts)
print(f'\nClass imbalance ratio: {target_counts[0]/target_counts[1]:.1f}:1')

fig, ax = plt.subplots(figsize=(5, 3))
target_counts.plot(kind='bar', ax=ax, color=['#2A9D8F', '#E63946'])
ax.set_xticklabels(['Non-vulnerable (0)', 'Vulnerable (1)'], rotation=0)
ax.set_title('Target Distribution')
plt.tight_layout()
plt.show()

## 3. Missing Value Analysis

In [ ]:
missing = (
    app.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame('missing_count')
)
missing['missing_pct'] = missing['missing_count'] / len(app) * 100
missing = missing[missing['missing_count'] > 0]
print(f'{len(missing)} columns have missing values')
missing.head(20)

In [ ]:
# Visualise top 30 missing
top_missing = missing.head(30)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_missing.index, top_missing['missing_pct'], color='#F4A261')
ax.set_xlabel('Missing %')
ax.set_title('Top 30 Features by Missing Rate')
ax.axvline(50, ls='--', color='red', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. EXT_SOURCE Features (Key Predictors)

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ext_cols):
    for target_val, colour, label in [(0, '#2A9D8F', 'Non-vulnerable'), (1, '#E63946', 'Vulnerable')]:
        subset = app[app['TARGET'] == target_val][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, color=colour, label=label, density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('EXT_SOURCE Distributions by Target', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Financial Ratio Analysis

In [ ]:
app['credit_to_income'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
app['annuity_to_income'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in [
    (axes[0], 'credit_to_income', 'Credit-to-Income Ratio'),
    (axes[1], 'annuity_to_income', 'Annuity-to-Income Ratio'),
]:
    for target_val, colour, label in [(0, '#2A9D8F', 'Non-vulnerable'), (1, '#E63946', 'Vulnerable')]:
        subset = app[app['TARGET'] == target_val][col].dropna()
        subset = subset[subset < subset.quantile(0.99)]  # clip outliers
        ax.hist(subset, bins=40, alpha=0.5, color=colour, label=label, density=True)
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Protected Attribute Analysis (FCA Fairness)

In [ ]:
# Vulnerability rate by gender
gender_rates = (
    app.groupby('CODE_GENDER')['TARGET']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'vuln_rate', 'count': 'n'})
)
print('Vulnerability rate by gender:')
print(gender_rates)

# Quick DIR check
rates = gender_rates['vuln_rate']
dir_val = rates.min() / rates.max()
print(f'\nDisparate Impact Ratio (raw data): {dir_val:.3f}')
print('FCA 4/5ths threshold: 0.80')
print('Status:', 'PASS ✅' if dir_val >= 0.80 else 'FAIL ❌')

## 7. Temporal Structure

Important for out-of-time (OOT) validation design.

In [ ]:
# DAYS_BIRTH proxy for customer age distribution
app['age_years'] = -app['DAYS_BIRTH'] / 365

fig, ax = plt.subplots(figsize=(8, 4))
for target_val, colour, label in [(0, '#2A9D8F', 'Non-vulnerable'), (1, '#E63946', 'Vulnerable')]:
    subset = app[app['TARGET'] == target_val]['age_years']
    ax.hist(subset, bins=30, alpha=0.5, color=colour, label=label, density=True)
ax.set_xlabel('Customer Age (years)')
ax.set_title('Age Distribution by Vulnerability')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Correlation with Target

In [ ]:
numeric_cols = app.select_dtypes(include='number').columns.tolist()
corr_with_target = (
    app[numeric_cols]
    .corr()['TARGET']
    .drop('TARGET')
    .sort_values(key=abs, ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(8, 6))
colours = ['#E63946' if v > 0 else '#2A9D8F' for v in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colours)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Top 20 Features Correlated with TARGET')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

## 9. Key Takeaways

Fill this in after running the cells above.

1. **Class imbalance**: ~X:1 majority:minority — SMOTE required during training.
2. **EXT_SOURCE features**: Strong predictors (negative correlation with risk — higher score = less risky).
3. **Missing data strategy**:
   - `EXT_SOURCE_*`: Use sentinel + indicator flag (not simple imputation).
   - `DAYS_EMPLOYED=365243`: Sentinel for unemployed — flag and replace.
4. **Protected attributes**: Initial DIR on raw data = X.XX — monitor closely in model output.
5. **Financial ratios**: Credit-to-income and annuity-to-income separate classes — strong engineered features.

---
*Next step: run `python training/train.py` to build the full pipeline.*